# H3 Supplemental DoS Operational Validation

**Research Question 3:** How effective are cybersecurity analytics in reducing denial of service risks in baggage tracking?

**Hypothesis 3:** Cybersecurity analytics reduces the operational impact of simulated denial of service conditions in RFID baggage tracking environments.

## Purpose

This experiment extends the original H3 simulation without replacing its results.

The analysis uses recorded CICIDS2017 network traffic and researcher generated RFID baggage tracking data.

Three conditions will be evaluated: no mitigation, always active store and forward, and analytics triggered store and forward.

**BLUF:** The objective is to measure whether analytics contributes to reducing modeled baggage tracking disruption beyond the recovery provided by buffering alone.

The public network records are not airport baggage observations. Any relationship between network detection and baggage disruption is a documented simulation assumption.

## Step 1: Validate the RFID Disruption Population

Identify the original simulated DoS records and verify the event fields, disruption durations, and affected baggage journeys before applying new experimental conditions.


In [3]:

from pathlib import Path
import pandas as pd

# STEP 1: Find the main research project folder
# Jupyter may start inside 00_Notebooks instead of the project root.

current_folder = Path.cwd().resolve()

print("Current working directory:", current_folder)

# Search the current folder and its parent directories
project_folder = None

for folder in [current_folder, *current_folder.parents]:
    if (
        (folder / "00_Notebooks").is_dir()
        and (folder / "06_Prepared_Data").is_dir()
    ):
        project_folder = folder
        break

if project_folder is None:
    raise FileNotFoundError(
        "Could not locate the main research project folder."
    )

print("Research project folder:", project_folder)

# STEP 2: Locate the prepared RFID dataset
data_folder = project_folder / "06_Prepared_Data"

matches = sorted(
    data_folder.rglob("02_Prepared_RFID_Data.csv")
)

# Display available CSV files if the expected name differs
if len(matches) != 1:
    print("\nAvailable prepared CSV files:")

    for file in sorted(data_folder.rglob("*.csv")):
        print(file.relative_to(project_folder))

    raise FileNotFoundError(
        f"Expected one prepared RFID dataset. Found {len(matches)}."
    )

file = matches[0]

# STEP 3: Load the original prepared RFID data
h3_data = pd.read_csv(file, low_memory=False)

print("\n=== ORIGINAL RFID DATA ===")
print("Dataset:", file)
print("Total records:", len(h3_data))

# STEP 4: Validate the required H3 fields
required = [
    "scan_id",
    "timestamp",
    "disruption_window",
    "disruption_minutes"
]

missing = [
    name for name in required
    if name not in h3_data.columns
]

if missing:
    raise ValueError(
        f"Missing required fields: {missing}"
    )

# STEP 5: Identify the original simulated DoS events
disruption = pd.to_numeric(
    h3_data["disruption_window"],
    errors="coerce"
)

dos_events = h3_data.loc[
    disruption.eq(1)
].copy()

print("\n=== H3 DISRUPTION POPULATION ===")
print("Known DoS events:", len(dos_events))
print(
    "Unique affected scans:",
    dos_events["scan_id"].nunique()
)

# Review baggage journey and disruption identifiers
for field in ["journey_id", "disruption_id"]:
    if field in dos_events.columns:
        print(
            f"Unique {field}:",
            dos_events[field].nunique()
        )

# STEP 6: Examine disruption durations
durations = pd.to_numeric(
    dos_events["disruption_minutes"],
    errors="coerce"
)

print("\n=== DISRUPTION DURATION ===")
print(durations.describe().round(2))

print("\n=== DATA QUALITY ===")
print(
    "Missing disruption durations:",
    durations.isna().sum()
)

print(
    "Duplicate scan identifiers:",
    dos_events["scan_id"].duplicated().sum()
)

print(
    "\nOriginal H3 event count matches:",
    len(dos_events) == 75
)

print("\nStep 1 complete.")
print("No source files were modified.")


Current working directory: /workspaces/Aviation-Baggage-Cybersecurity-Research/00_Notebooks
Research project folder: /workspaces/Aviation-Baggage-Cybersecurity-Research

=== ORIGINAL RFID DATA ===
Dataset: /workspaces/Aviation-Baggage-Cybersecurity-Research/06_Prepared_Data/02_Prepared_RFID_Data.csv
Total records: 31575

=== H3 DISRUPTION POPULATION ===
Known DoS events: 75
Unique affected scans: 75
Unique disruption_id: 10

=== DISRUPTION DURATION ===
count    75.00
mean     20.81
std       4.31
min      16.00
25%      17.00
50%      20.00
75%      23.00
max      29.00
Name: disruption_minutes, dtype: float64

=== DATA QUALITY ===
Missing disruption durations: 0
Duplicate scan identifiers: 0

Original H3 event count matches: True

Step 1 complete.
No source files were modified.
